# Ungraded Lab: Pipeline Lab

## Task 1: Pipeline Setup and Configuration

<b>Steps:</b> 

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime
import boto3
import logging
from io import StringIO
from botocore.exceptions import ClientError

# Your code here:
# 1. Upload ticketwise_dataset.csv (Navigate to Desktop / Datasets / ticketwise_dataset.csv) to an S3 bucket. 
# This bucket will be used throughout this lab
# 2. Set up logging configuration
# 3. Initialize AWS S3 client
# 4. Define pipeline constants (bucket names, required columns, etc.)

## Task 2: Enhanced Data Ingestion

<b>Steps:</b> 

In [ ]:
def validate_ticket_data(df):
    """
    Implement comprehensive ticket validation
    
    Parameters:
        df: DataFrame containing ticket data
    
    Returns:
        tuple: (is_valid, error_messages)
    """
    # Your code here:
    # 1. Check required columns
    # 2. Validate data types
    # 3. Check value ranges
    # 4. Return validation results

def ingest_and_validate(source_path):
    """
    Ingest and validate ticket data
    
    Parameters:
        source_path: Path to source data
    
    Returns:
        DataFrame: Validated ticket data
    """
    # Your code here:
    # 1. Load data
    # 2. Run validation
    # 3. Log validation results
    # 4. Return validated data

## Task 3: Intelligent Data Processing

<b>Steps:</b> 

In [ ]:
def calculate_priority_score(row):
    """
    Calculate ticket priority score based on multiple factors
    
    Parameters:
        row: DataFrame row containing ticket information
    
    Returns:
        float: Priority score
    """
    # Your code here:
    # 1. Consider VIP status
    # 2. Factor in contract value
    # 3. Account for urgency flags
    # 4. Return weighted score

def process_ticket_data(df):
    """
    Process and enhance ticket data
    
    Parameters:
        df: DataFrame with raw ticket data
    
    Returns:
        DataFrame: Processed and enhanced data
    """
    # Your code here:
    # 1. Clean and standardize fields
    # 2. Calculate priority scores
    # 3. Add derived features
    # 4. Implement AI-suggested transformations

## Task 4: Organized Storage Implementation


<b>Steps:</b> 

In [ ]:
def create_storage_structure(bucket_name):
    """
    Create organized storage structure in S3
    
    Parameters:
        bucket_name: Name of S3 bucket
    """
    # Your code here:
    # 1. Define folder structure
    # 2. Create necessary paths
    # 3. Set up metadata

def store_processed_tickets(df, bucket_name):
    """
    Store processed tickets in organized structure
    
    Parameters:
        df: Processed ticket DataFrame
        bucket_name: Target S3 bucket
    """
    # Your code here:
    # 1. Organize by priority
    # 2. Include metadata
    # 3. Implement versioning
    # 4. Log storage operations

## Task 5:  Pipeline Integration and Monitoring

<b>Steps:</b> 

In [ ]:
def run_pipeline(source_path, bucket_name):
    """
    Execute complete pipeline with monitoring
    
    Parameters:
        source_path: Source data location
        bucket_name: Target S3 bucket
    """
    # Your code here:
    # 1. Initialize monitoring
    # 2. Run pipeline stages
    # 3. Track performance metrics
    # 4. Generate pipeline report

## Solution Code
Need a hand or are curious to compare your approach? Below is a complete solution you can use as a reference. This is just one of many valid ways to solve the problem. Make sure to give it a try on your own first. Use this implementation to troubleshoot, learn new techniques, or confirm your logic. Keep experimenting and enjoy the process!

## Task 1: Pipeline Setup and Configuration Solution Code

<b>Steps:</b> 

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import boto3
import logging
from io import StringIO
from botocore.exceptions import ClientError

# 1. Upload ticketwise_dataset.csv to your S3 bucket manually
# (Navigate to Desktop > Datasets > ticketwise_dataset.csv and upload it)
BUCKET_NAME = "ticketwise-pipeline"  # replace with your bucket
FILE_KEY = "ticketwise_dataset.csv"

# 2. Set up logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# 3. Initialize AWS S3 client
s3_client = boto3.client('s3')

# 4. Define pipeline constants
REQUIRED_COLUMNS = ['submission_time', 'channel', 'issue_type']
TODAY = datetime.now().strftime('%Y-%m-%d')

logger.info("Setup complete. Ready to run the pipeline.")

## Task 2: Enhanced Data Ingestion Solution Code

<b>Steps:</b> 

In [ ]:
def validate_ticket_data(df):
    """
    Implement comprehensive ticket validation
    
    Parameters:
        df: DataFrame containing ticket data
    
    Returns:
        tuple: (is_valid, error_messages)
    """
    error_messages = []
    required_columns = ['submission_time', 'channel', 'issue_type']
    
    # 1. Check required columns
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        error_messages.append(f"Missing required columns: {missing_columns}")
    
    # 2. Validate data types
    if 'submission_time' in df.columns:
        # Convert submission_time to datetime
        df['submission_time'] = pd.to_datetime(df['submission_time'])

        if not np.issubdtype(df['submission_time'].dtype, np.datetime64):
            error_messages.append("submission_time must be datetime")
    
    if 'channel' in df.columns:
        if not np.issubdtype(df['channel'].dtype, np.object_):
            error_messages.append("channel must be string")
    
    # 3. Check value ranges / valid values
    if 'issue_type' in df.columns:
        invalid_values = df[~df['issue_type'].notna() & (df['issue_type'] == '')]
        if len(invalid_values) > 0:
            error_messages.append("issue_type contains empty values")
    
    # 4. Return validation results
    is_valid = len(error_messages) == 0
    
    return is_valid, error_messages

def ingest_and_validate(bucket_name, file_key):
    """
    Ingest and validate ticket data from S3
    
    Parameters:
        bucket_name: Name of the S3 bucket
        file_key: S3 object key (file path)
    
    Returns:
        DataFrame: Validated ticket data
    """
    try:
        # Initialize S3 client
        s3_client = boto3.client('s3')
        
        # 1. Load data from S3
        response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
        content = response['Body'].read().decode('utf-8')
        df = pd.read_csv(StringIO(content))
        logger.info(f"Loaded {len(df)} records from s3://{bucket_name}/{file_key}")
        
        # 2. Run validation
        is_valid, errors = validate_ticket_data(df)
        
        # 3. Log validation results
        if is_valid:
            logger.info("Data validation passed")
        else:
            logger.error(f"Data validation failed with errors: {errors}")
            raise ValueError("Validation failed")
        
        # 4. Return validated data
        return df
    
    except ClientError as e:
        logger.error(f"AWS error: {e}")
        raise
    except Exception as e:
        logger.error(f"Ingestion and validation failed: {str(e)}")
        raise

validated_df = ingest_and_validate(BUCKET_NAME, FILE_KEY)

## Task 3: Intelligent Data Processing Solution Code

<b>Steps:</b> 

In [ ]:
def calculate_priority_score(row):
    """
    Calculate ticket priority score based on multiple factors
    
    Parameters:
        row: DataFrame row containing ticket information
    
    Returns:
        float: Priority score
    """
    score = 0.0
    
    # 1. Consider VIP status
    if row.get('is_vip', False):
        score += 5
    
    # 2. Factor in contract value (higher value = higher priority)
    contract_value = row.get('contract_value', 0)
    if contract_value > 5000:
        score += 3
    elif contract_value > 1000:
        score += 2
    elif contract_value > 0:
        score += 1
    
    # 3. Account for urgency flags
    if row.get('self_declared_p1', False):
        score += 4
    
    # 4. Return weighted score
    return score


def process_ticket_data(df):
    """
    Process and enhance ticket data
    
    Parameters:
        df: DataFrame with raw ticket data
    
    Returns:
        DataFrame: Processed and enhanced data
    """
    try:
        logger.info("Starting ticket data processing")
        
        # 1. Clean and standardize fields
        df['submission_time'] = pd.to_datetime(df['submission_time'])
        df['channel'] = df['channel'].str.lower().str.strip()
        df['issue_type'] = df['issue_type'].fillna('Other')
        
        # 2. Calculate priority scores
        if all(col in df.columns for col in ['is_vip', 'contract_value', 'self_declared_p1']):
            df['priority_score'] = df.apply(calculate_priority_score, axis=1)
        else:
            df['priority_score'] = 0  # default if columns are missing
        
        # 3. Add derived features
        df['hour_of_day'] = df['submission_time'].dt.hour
        df['is_business_hours'] = df['hour_of_day'].between(9, 17)
        
        # 4. Implement AI-suggested transformations (example: flag critical issues)
        df['critical_issue_flag'] = df['issue_type'].str.contains('critical', case=False, na=False)
        
        logger.info("Ticket data processing completed")
        return df
    
    except Exception as e:
        logger.error(f"Processing failed: {str(e)}")
        raise

validated_df_processed = process_ticket_data(validated_df)

## Task 4: Organized Storage Implementation Solution Code

<b>Steps:</b> 

In [ ]:
def create_storage_structure(bucket_name):
    """
    Create organized storage structure in S3
    
    Parameters:
        bucket_name: Name of S3 bucket
    """
    try:
        # Define folder structure (example: by year/month/day)
        today = datetime.now()
        structure = f"{today.year}/{today.month:02}/{today.day:02}/"
        
        # Create a dummy file in the structure to ensure folders exist
        s3_client.put_object(Bucket=bucket_name, Key=f"{structure}placeholder.txt", Body="")
        logger.info(f"Storage structure created: s3://{bucket_name}/{structure}")
        return structure
    
    except ClientError as e:
        logger.error(f"Failed to create storage structure: {str(e)}")
        raise

def store_processed_tickets(df, bucket_name):
    """
    Store processed tickets in organized structure
    
    Parameters:
        df: Processed ticket DataFrame
        bucket_name: Target S3 bucket
    """
    try:
        # 1. Create storage structure
        folder_path = create_storage_structure(bucket_name)
        
        # 2. Organize by priority (example: high/medium/low)
        for priority in df['priority_score'].unique():
            subset = df[df['priority_score'] == priority]
            file_key = f"{folder_path}priority_{priority}_{datetime.now().strftime('%H%M%S')}.csv"
            
            # 3. Include metadata (example: number of records)
            metadata = {'record_count': str(len(subset))}
            
            # 4. Upload to S3
            csv_buffer = StringIO()
            subset.to_csv(csv_buffer, index=False)
            s3_client.put_object(
                Bucket=bucket_name,
                Key=file_key,
                Body=csv_buffer.getvalue(),
                Metadata=metadata
            )
            logger.info(f"Stored {len(subset)} records at s3://{bucket_name}/{file_key}")
    
    except ClientError as e:
        logger.error(f"S3 operation failed: {str(e)}")
        raise
    except Exception as e:
        logger.error(f"Storing processed tickets failed: {str(e)}")
        raise

store_processed_tickets(validated_df_processed, BUCKET_NAME)

## Task 5: Pipeline Integration and Monitoring Solution Code

<b>Steps:</b> 

In [ ]:
def run_pipeline(source_path, bucket_name):
    """
    Execute complete pipeline with monitoring
    
    Parameters:
        source_path: Source data location
        bucket_name: Target S3 bucket
    """
    import time
    start_time = time.time()
    
    try:
        logger.info("Pipeline execution started")
        
        # 1. Initialize monitoring (example: track start time)
        pipeline_metrics = {'start_time': datetime.now()}
        
        # 2. Run pipeline stages
        validated_df = ingest_and_validate(BUCKET_NAME, FILE_KEY)
        processed_df = process_ticket_data(validated_df)
        store_processed_tickets(processed_df, BUCKET_NAME)
        
        # 3. Track performance metrics
        pipeline_metrics['end_time'] = datetime.now()
        pipeline_metrics['duration_seconds'] = time.time() - start_time
        pipeline_metrics['records_processed'] = len(processed_df)
        
        # 4. Generate pipeline report
        logger.info(f"Pipeline completed successfully in {pipeline_metrics['duration_seconds']:.2f} seconds")
        logger.info(f"Records processed: {pipeline_metrics['records_processed']}")
        logger.info(f"Pipeline metrics: {pipeline_metrics}")
        
    except Exception as e:
        logger.error(f"Pipeline failed: {str(e)}")
        raise
run_pipeline(FILE_KEY, BUCKET_NAME)